In [1]:
%pip install -U sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Install required packages
%pip install -U FlagEmbedding pandas openpyxl

# Run the grant ranking script
#!python rank_grants.py "Other Languages.xlsx" \
 #   --profile "We are a women-led NGO in Armenia focused on gender equality and women's rights"

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------
# 1. LOAD EXCEL DATA
# ---------------------------------------

FILE_PATH = "Copy of Ardook_Data_All_Grant_Recommendations_UPDATED_v3.xlsx.xlsm"

df = pd.read_excel(
    FILE_PATH,
    sheet_name="Grant Tracker"
)

# ---------------------------------------
# 2. LOAD SENTENCE TRANSFORMER
# ---------------------------------------

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# ---------------------------------------
# 3. DEFINE SEARCH CRITERIA
# ---------------------------------------

search_queries = [
    "Grants for female founders and female entrepreneurs",
    "Funding for women-owned businesses",
    "Grants for women-led startups",
    "Funding for women in technology and AI",
    "Grants for small business owners",
    "Funding for small and independently owned businesses",
    "Grants for female-owned technology companies",
    "Funding for mothers who own businesses",
    "Grants supporting women entrepreneurs and business development"
]

# Combine relevant grant information
text_columns = [
    "Program",
    "Type",
    "Core Requirements",
    "Why It Fits Ardook Data",
    "Key Concern / Disqualifier"
]

df["Combined_Text"] = (
    df[text_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)

# ---------------------------------------
# 4. GENERATE EMBEDDINGS
# ---------------------------------------

grant_embeddings = model.encode(
    df["Combined_Text"].tolist(),
    normalize_embeddings=True
)

query_embeddings = model.encode(
    search_queries,
    normalize_embeddings=True
)

# ---------------------------------------
# 5. CALCULATE SIMILARITY
# ---------------------------------------

similarity_matrix = cosine_similarity(
    grant_embeddings,
    query_embeddings
)

# Average similarity across all search queries
df["Similarity_Score"] = similarity_matrix.mean(axis=1)

# Highest individual query similarity
df["Best_Match_Score"] = similarity_matrix.max(axis=1)

# ---------------------------------------
# 6. KEYWORD DETECTION
# ---------------------------------------

keywords = [
    "female",
    "women",
    "woman",
    "women-owned",
    "woman-owned",
    "women-led",
    "female founder",
    "female-owned",
    "small business",
    "small-business",
    "small business owner",
    "entrepreneur",
    "mother"
]

def find_keywords(text):
    text = str(text).lower()

    matches = [
        keyword for keyword in keywords
        if keyword in text
    ]

    return ", ".join(matches)

df["Matched_Keywords"] = df["Combined_Text"].apply(
    find_keywords
)

df["Keyword_Count"] = df["Matched_Keywords"].apply(
    lambda x: len(x.split(", ")) if x else 0
)

# ---------------------------------------
# 7. CREATE FINAL RELEVANCE SCORE
# ---------------------------------------

# Similarity is the primary ranking signal.
# Keyword coverage provides an additional signal.

df["Relevance_Score"] = (
    df["Similarity_Score"] * 80
    + np.minimum(df["Keyword_Count"], 5) * 4
)

# ---------------------------------------
# 8. IDENTIFY APPLICATION ELIGIBILITY
# ---------------------------------------

def check_status(row):
    status = str(
        row["Current Status (as last reported)"]
    ).lower()

    grant_type = str(row["Type"]).lower()

    # Exclude closed opportunities
    if "closed" in status:
        return "Closed"

    # Exclude programs without verified grant funding
    if any(word in grant_type for word in [
        "accelerator",
        "pitch platform",
        "competition",
        "challenge"
    ]):
        if "grant" not in grant_type and "funding" not in grant_type:
            return "Review - Not Confirmed Grant"

    if any(word in status for word in [
        "watchlist",
        "monitor",
        "not verified",
        "coming soon"
    ]):
        return "Watchlist"

    return "Potentially Applicable"

df["Application_Status"] = df.apply(
    check_status,
    axis=1
)

# ---------------------------------------
# 9. IDENTIFY BEST MATCHES
# ---------------------------------------

# High relevance threshold: 0.45
# This is an initial screening threshold,
# not a probability of eligibility.

df["Strong_Match"] = (
    df["Best_Match_Score"] >= 0.45
)

# Count relevant opportunities
best_grants = df[
    (df["Strong_Match"] == True)
    & (df["Application_Status"] == "Potentially Applicable")
]

# ---------------------------------------
# 10. RANK TOP 5 GRANTS
# ---------------------------------------

ranked_grants = best_grants.sort_values(
    by=["Relevance_Score", "Best_Match_Score"],
    ascending=False
)

top_5 = ranked_grants.head(5).copy()

top_5.insert(
    0,
    "Rank",
    range(1, len(top_5) + 1)
)

# ---------------------------------------
# 11. DISPLAY RESULTS
# ---------------------------------------

print("\n========== GRANT ANALYSIS ==========")

print("Total grants analyzed:", len(df))

print(
    "Highly relevant, potentially applicable grants:",
    len(best_grants)
)

print(
    "Total strong semantic matches:",
    int(df["Strong_Match"].sum())
)

print("\n========== TOP 5 GRANTS ==========")

for _, grant in top_5.iterrows():

    print(f"\nRank: {grant['Rank']}")
    print("Program:", grant["Program"])
    print("Amount:", grant["Amount / Benefit"])
    print("Deadline:", grant["Deadline"])
    print("Similarity:", round(grant["Best_Match_Score"], 3))
    print("Keywords:", grant["Matched_Keywords"])
    print("Why it fits:", grant["Why It Fits Ardook Data"])
    print("Requirements:", grant["Core Requirements"])
    print("Concern:", grant["Key Concern / Disqualifier"])
    print("Source:", grant["Official Source"])

# ---------------------------------------
# 12. EXPORT RESULTS TO EXCEL
# ---------------------------------------

with pd.ExcelWriter(
    "Grant_Similarity_Analysis.xlsx",
    engine="openpyxl"
) as writer:

    df.sort_values(
        by="Relevance_Score",
        ascending=False
    ).to_excel(
        writer,
        sheet_name="All Grant Rankings",
        index=False
    )

    top_5.to_excel(
        writer,
        sheet_name="Top 5 Grants",
        index=False
    )

    summary = pd.DataFrame({
        "Metric": [
            "Total Opportunities",
            "Strong Semantic Matches",
            "Potentially Applicable Strong Matches",
            "Closed Opportunities",
            "Watchlist Opportunities"
        ],
        "Count": [
            len(df),
            int(df["Strong_Match"].sum()),
            len(best_grants),
            (df["Application_Status"] == "Closed").sum(),
            (df["Application_Status"] == "Watchlist").sum()
        ]
    })

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

print("\nAnalysis complete!")
print("Results saved to Grant_Similarity_Analysis.xlsx")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


========== GRANT ANALYSIS ==========
Total grants analyzed: 37
Highly relevant, potentially applicable grants: 14
Total strong semantic matches: 24

========== TOP 5 GRANTS ==========

Rank: 1
Program: WomensNet Amber Grant / Startup Grant
Amount: $10,000 monthly; monthly winners may qualify for $50,000 year-end award
Deadline: 2026-09-30
Similarity: 0.746
Keywords: women, woman, women-owned
Why it fits: Good early-stage woman-founder fit
Requirements: Women-owned business; Startup Grant track for idea/minimal-sales businesses
Concern: Generic rather than AI-specific
Source: https://ambergrantsforwomen.com/get-an-amber-grant/

Rank: 2
Program: Giving Joy Grant
Amount: Up to $500
Deadline: 2026-09-30
Similarity: 0.71
Keywords: women, entrepreneur
Why it fits: Social-impact/women angle
Requirements: Women entrepreneurs; proposed activity must benefit women, girls and/or families
Concern: Very small award relative to founder time
Source: https://givingjoygrants.org/

Rank: 3
Program: Her

In [5]:
import pandas as pd
from FlagEmbedding import BGEM3FlagModel
from IPython.display import display

In [6]:
# Input spreadsheet
input_file = "Other_Languages.xlsx"

# Output spreadsheet
output_file = "ranked_grants.xlsx"

# Organization profile
profile = """
We are a women-led NGO in Armenia focused on gender equality,
women's rights, women's empowerment, and supporting women and
girls through social, educational, and community-based initiatives.
"""

# BGE-M3 hybrid scoring weights
WEIGHTS = [0.4, 0.2, 0.4]

# Settings
USE_AMOUNT = False
TOP_N = 0  # 0 = include all grants

In [7]:
# Load Excel spreadsheet
df = pd.read_excel(input_file)

# Check required columns
required_columns = ["Name", "Requirements", "Amount", "Link"]

missing_columns = set(required_columns) - set(df.columns)

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

# Remove rows without grant names or requirements
df = df.dropna(
    subset=["Name", "Requirements"],
    how="all"
).reset_index(drop=True)

print(f"Total grants loaded: {len(df)}")

display(df.head())

Total grants loaded: 3


,Name,Requirements,Amount,Link
0,Կանչ կանանց նորաստեղծ ՔՀԿ-ների համար. Ինստիտու...,"Լինեն կանանց կողմից ղեկավարվող ՔՀԿ, գրանցված Հ...",1 748 000 ՀՀ դրամ,https://womenofarmenia.org/2026/02/23/call-for...
1,Կանանց հիմնադրամ Հայաստանում,Տարբեր հարցերի շուրջ 40 անց կանանց ձայնը լսելի...,1 միլիոն 200 հազար ՀՀ,https://womenfundarmenia.org/%D5%A4%D6%80%D5%A...
2,For Her Future,ინიციატივა დაეხმარება საწარმოებს დანერგონ ინოვ...,"40, 000 ლარი",https://forher.ge/granti/mtsire-da-sashualo-sa...


In [8]:
def build_doc_text(row, use_amount=False):

    parts = [
        str(row["Name"]).strip(),
        str(row["Requirements"]).strip()
    ]

    if use_amount:
        parts.append(str(row["Amount"]).strip())

    # Remove empty or missing values
    parts = [
        p for p in parts
        if p and p.lower() != "nan"
    ]

    return "\n".join(parts)


# Create grant descriptions
docs = [
    build_doc_text(row, USE_AMOUNT)
    for _, row in df.iterrows()
]

print("Grant descriptions prepared:", len(docs))

# Preview one grant
print(docs[0])

Grant descriptions prepared: 3
Կանչ կանանց նորաստեղծ ՔՀԿ-ների համար. Ինստիտուցիոնալ հզորացման դրամաշնորհներ (3-6 ամիս)
Լինեն կանանց կողմից ղեկավարվող ՔՀԿ, գրանցված Հայաստանում վերջին 1–3 տարիների ընթացքում
Գործեն գենդերային հավասարության, կանանց իրավունքների կամ ինտերսեկցիոնալ արդարության ոլորտում
Ցուցաբերեն հանձնառություն մարդու իրավունքների և հաշվետվողականության սկզբունքներին


In [9]:
model = BGEM3FlagModel(
    "BAAI/bge-m3",
    use_fp16=True
)

print("BGE-M3 model loaded successfully!")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BGE-M3 model loaded successfully!


In [10]:
# Create profile-grant pairs
pairs = [
    [profile, grant]
    for grant in docs
]

# Calculate similarity scores
scores = model.compute_score(
    pairs,
    max_passage_length=2048,
    weights_of_each_score=WEIGHTS
)

print("Similarity scoring completed!")

Similarity scoring completed!


In [11]:
# Add individual similarity scores
df["score_dense"] = scores["dense"]
df["score_sparse"] = scores["sparse"]
df["score_colbert"] = scores["colbert"]

# Add combined hybrid score
df["score"] = scores["colbert+sparse+dense"]

# Sort grants from highest to lowest similarity
ranked = df.sort_values(
    by="score",
    ascending=False
).reset_index(drop=True)

# Add ranking column
ranked.insert(0, "Rank", ranked.index + 1)

# Optionally display only the top N grants
if TOP_N > 0:
    ranked = ranked.head(TOP_N)

display(ranked.head(20))

,Rank,Name,Requirements,Amount,Link,score_dense,score_sparse,score_colbert,score
0,1,Կանանց հիմնադրամ Հայաստանում,Տարբեր հարցերի շուրջ 40 անց կանանց ձայնը լսելի...,1 միլիոն 200 հազար ՀՀ,https://womenfundarmenia.org/%D5%A4%D6%80%D5%A...,0.540527,0.000000,0.578470,0.372999
1,2,Կանչ կանանց նորաստեղծ ՔՀԿ-ների համար. Ինստիտու...,"Լինեն կանանց կողմից ղեկավարվող ՔՀԿ, գրանցված Հ...",1 748 000 ՀՀ դրամ,https://womenofarmenia.org/2026/02/23/call-for...,0.535645,0.000000,0.501764,0.345803
2,3,For Her Future,ინიციატივა დაეხმარება საწარმოებს დანერგონ ინოვ...,"40, 000 ლარი",https://forher.ge/granti/mtsire-da-sashualo-sa...,0.503906,0.010773,0.505671,0.340117


In [12]:
# Export ranked grants
ranked.to_excel(
    output_file,
    index=False
)

print(f"Ranked grants successfully saved to: {output_file}")

Ranked grants successfully saved to: ranked_grants.xlsx


In [13]:
%pip install deep-translator

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [14]:
from deep_translator import GoogleTranslator
import pandas as pd
import time
import random

translator = GoogleTranslator(source="auto", target="en")

def translate_one(text, max_retries=5):
    """Translate a single string with exponential backoff on rate-limit errors."""
    for attempt in range(max_retries):
        try:
            return translator.translate(text)
        except Exception as e:
            wait = (2 ** attempt) + random.uniform(0, 1)
            print(f"Retry {attempt + 1}/{max_retries} in {wait:.1f}s ({str(e)[:60]}...)")
            time.sleep(wait)
    return text  # fall back to original after all retries fail


def translate_column(series, delay=0.4, cache=None):
    """Translate a column one request at a time, ~2.5 requests/sec, with caching."""
    cache = {} if cache is None else cache
    results = []

    for text in series.fillna("").astype(str):
        text = text.strip()

        # Skip empty strings and avoid re-translating duplicates
        if not text:
            results.append("")
            continue
        if text in cache:
            results.append(cache[text])
            continue

        # Google Translate has a ~5000 char limit per request
        translated = translate_one(text[:4900])
        cache[text] = translated
        results.append(translated)

        time.sleep(delay)  # stay well under 5 requests/sec

    return results


ranked_english = ranked.copy()
cache = {}

ranked_english["Name_English"] = translate_column(ranked_english["Name"], cache=cache)
ranked_english["Requirements_English"] = translate_column(ranked_english["Requirements"], cache=cache)

Retry 1/5 in 1.3s (Server Error: You made too many requests to the server.Accor...)
Retry 2/5 in 3.0s (Server Error: You made too many requests to the server.Accor...)
Retry 3/5 in 4.6s (Server Error: You made too many requests to the server.Accor...)


KeyboardInterrupt: 

In [15]:
from deep_translator import GoogleTranslator
print(GoogleTranslator(source="auto", target="en").translate("Bonjour"))

TooManyRequests: Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function

In [16]:
%pip install -U transformers sentencepiece langdetect

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [17]:
import torch
import pandas as pd

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langdetect import detect

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)

translation_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

print("Translation model loaded successfully!")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (4 outputs left out)

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Translation model loaded successfully!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

In [21]:
import re
import torch
import pandas as pd

# NLLB language codes
LANGUAGE_CODES = {
    "Armenian": "hye_Armn",
    "Georgian": "kat_Geor",
    "English": "eng_Latn",
    "Russian": "rus_Cyrl",
    "French": "fra_Latn",
    "Spanish": "spa_Latn",
    "German": "deu_Latn"
}


# Detect Armenian and Georgian using their Unicode characters
def detect_source_language(text):

    armenian_count = sum(
        0x0530 <= ord(char) <= 0x058F
        for char in text
    )

    georgian_count = sum(
        0x10A0 <= ord(char) <= 0x10FF
        or 0x2D00 <= ord(char) <= 0x2D2F
        for char in text
    )

    if armenian_count > georgian_count and armenian_count > 0:
        return "Armenian"

    if georgian_count > armenian_count and georgian_count > 0:
        return "Georgian"

    # Use automatic detection for other languages
    from langdetect import detect

    detected = detect(text)

    language_map = {
        "en": "English",
        "ru": "Russian",
        "fr": "French",
        "es": "Spanish",
        "de": "German"
    }

    return language_map.get(detected, "Unknown")


# Translate text into English
def translate_to_english(text):

    if pd.isna(text) or str(text).strip() == "":
        return text

    text = str(text).strip()

    try:
        source_language = detect_source_language(text)

        # No translation needed
        if source_language == "English":
            return text

        if source_language == "Unknown":
            print(f"Unsupported language: {text[:50]}")
            return text

        source_code = LANGUAGE_CODES[source_language]

        tokenizer.src_lang = source_code

        # Split long text into manageable token chunks
        token_ids = tokenizer(
            text,
            add_special_tokens=False
        )["input_ids"]

        chunks = [
            token_ids[i:i + 400]
            for i in range(0, len(token_ids), 400)
        ]

        translations = []

        for chunk in chunks:

            chunk_text = tokenizer.decode(
                chunk,
                skip_special_tokens=True
            )

            inputs = tokenizer(
                chunk_text,
                return_tensors="pt",
                truncation=True,
                max_length=512
            )

            device = next(translation_model.parameters()).device
            inputs = {
                key: value.to(device)
                for key, value in inputs.items()
            }

            with torch.no_grad():

                translated_tokens = translation_model.generate(
                    **inputs,
                    forced_bos_token_id=tokenizer.convert_tokens_to_ids(
                        "eng_Latn"
                    ),
                    max_length=512,
                    num_beams=4
                )

            translated_text = tokenizer.batch_decode(
                translated_tokens,
                skip_special_tokens=True
            )[0]

            translations.append(translated_text)

        return " ".join(translations)

    except Exception as e:
        print(f"Translation failed: {e}")
        return text

In [26]:
from tqdm.notebook import tqdm

# Only translate the top 10 grants
ranked_english = ranked.head(10).copy()

# Cache translations to avoid translating duplicate text
translation_cache = {}

def translate_cached(text):

    if pd.isna(text):
        return text

    text = str(text).strip()

    if not text:
        return text

    if text not in translation_cache:
        translation_cache[text] = translate_to_english(text)

    return translation_cache[text]


# Translate grant names
ranked_english["Name_English"] = [
    translate_cached(text)
    for text in tqdm(
        ranked_english["Name"],
        desc="Translating grant names"
    )
]

# Translate grant requirements
ranked_english["Requirements_English"] = [
    translate_cached(text)
    for text in tqdm(
        ranked_english["Requirements"],
        desc="Translating requirements"
    )
]

print("Translation completed!")

Translating grant names:   0%|          | 0/3 [00:00<?, ?it/s]

Translating requirements:   0%|          | 0/3 [00:00<?, ?it/s]

Translation completed!


In [27]:
display_columns = [
    "Rank",
    "Name_English",
    "Requirements_English",
    "Amount",
    "score_dense",
    "score_sparse",
    "score_colbert",
    "score",
    "Link"
]

display(ranked_english[display_columns])

,Rank,Name_English,Requirements_English,Amount,score_dense,score_sparse,score_colbert,score,Link
0,1,The Women's Foundation in Armenia,Activities for women over 40 about women's sex...,1 միլիոն 200 հազար ՀՀ,0.540527,0.000000,0.578470,0.372999,https://womenfundarmenia.org/%D5%A4%D6%80%D5%A...
1,2,Call for women's CSOs: Institutional empowerme...,Women-led NGOs registered in Armenia for the p...,1 748 000 ՀՀ դրամ,0.535645,0.000000,0.501764,0.345803,https://womenofarmenia.org/2026/02/23/call-for...
2,3,For Her Future,The initiative will help enterprises implement...,"40, 000 ლარი",0.503906,0.010773,0.505671,0.340117,https://forher.ge/granti/mtsire-da-sashualo-sa...


In [28]:
ranked_english.to_excel(
    "top_10_ranked_grants_english.xlsx",
    index=False
)

print("Export completed!")

Export completed!
